# Can we predict late deliveries?
Protocol: `protocol.yaml` (temporal holdout, 5-split walk-forward CV, choose by mean CV ROC AUC).

Feature decision: `claim_paid_eur` is compensation paid on a claim settled *after* delivery, so it is not known at dispatch (prediction time). It is excluded as leakage (a claim is paid on ~90% of late consignments and never otherwise). `consignment` (identifier) and `dispatched_on` (split only) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train = df.drop(index=hold_idx)
train = train.assign(_t=pd.to_datetime(train.dispatched_on)).sort_values('_t', kind='stable')
hold = df.loc[hold_idx]
print(len(train), len(hold), train.late_delivery.mean(), hold.late_delivery.mean())

In [ ]:
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT
Xtr, ytr = train[FEATURES], train.late_delivery
Xho, yho = hold[FEATURES], hold.late_delivery
def prep(): return ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
# all candidates are named before any is scored
candidates = {
 'baseline_prior': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'hgb_shallow': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'hgb_deeper': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
folds = list(TimeSeriesSplit(n_splits=5).split(Xtr))   # walk-forward on time-sorted training rows
train_pos = train.index.to_numpy()   # 0-based row positions in data.csv
cv_scores = {}
for name, est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(Xtr.iloc[tr],ytr.iloc[tr])   # preprocessing is inside the pipeline, fit per fold
        s.append(float(roc_auc_score(ytr.iloc[va], m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen = cv['mean'].idxmax(); print('chosen:', chosen)

In [ ]:
final = clone(candidates[chosen]).fit(Xtr,ytr)
holdout_auc = float(roc_auc_score(yho, final.predict_proba(Xho)[:,1]))
print('Headline holdout ROC AUC:', round(holdout_auc,4))

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(train_pos[i]) for i in tr], [int(train_pos[i]) for i in va]] for tr,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_idx],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The headline is the chosen candidate's single score on the holdout rows; the holdout was not used for any choice.